# Section 5.5 — Sensitivity to Temperature Variance

Reproduces the result table from section 5.5 of the thesis.
Re-runs the **10-cage** experiment (EEV, SP, DE, WS) with a **reduced-variance** scenario tree:
cold and warm branches are each pulled 1°C closer to normal,
halving the cold–warm spread from 4°C to 2°C.

| Branch | Jan | Feb | Mar | Apr | May | Jun | Jul | Aug | Sep | Oct | Nov | Dec |
|--------|-----|-----|-----|-----|-----|-----|-----|-----|-----|-----|-----|-----|
| Cold   | 4.0 | 4.0 | 4.0 | 5.0 | 8.0 | 11.0 | 13.0 | 15.5 | 14.5 | 12.0 | 9.0  | 6.5 |
| Normal | 5.0 | 5.0 | 5.0 | 6.0 | 9.0 | 12.0 | 14.0 | 16.5 | 15.5 | 13.0 | 10.0 | 7.5 |
| Warm   | 6.0 | 6.0 | 6.0 | 7.0 | 10.0 | 13.0 | 15.0 | 17.5 | 16.5 | 14.0 | 11.0 | 8.5 |

All other parameters (4-stage tree, 81 scenarios, survival coupling, 2% MIP gap) are unchanged.


In [12]:
import sys
import os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', 'models'))

import time
import numpy as np
import pandas as pd

from instance import (
    T, loc_mab, regional_mab,
    units_df as full_units_df,
    S_normal_v, S_bad_v,
    labels, stage_slices,
)
from IP import SalmonFarmingMILP
from DE import DeterministicEquivalent
from SP import BinaryProgressiveHedging


In [13]:
# Reduced-variance temperature profiles (cold/warm each 1°C closer to normal)
temps_cold_rv   = np.array([4.0,  4.0,  4.0,  5.0,  8.0,  11.0, 13.0, 15.5, 14.5, 12.0,  9.0,  6.5])
temps_normal_rv = np.array([5.0,  5.0,  5.0,  6.0,  9.0,  12.0, 14.0, 16.5, 15.5, 13.0, 10.0,  7.5])
temps_warm_rv   = np.array([6.0,  6.0,  6.0,  7.0, 10.0,  13.0, 15.0, 17.5, 16.5, 14.0, 11.0,  8.5])

# Verify: cold/warm are exactly 1°C from normal, normal is unchanged
assert np.allclose(temps_normal_rv - temps_cold_rv,  1.0)
assert np.allclose(temps_warm_rv   - temps_normal_rv, 1.0)

temp_map_rv = {'bad': temps_cold_rv, 'normal': temps_normal_rv, 'good': temps_warm_rv}

print('Temperature profiles verified.')


Temperature profiles verified.


In [14]:
# Parameterised scenario builder

def _tile(arr, n):
    return np.tile(arr, (n // 12) + 1)[:n]


def build_scenarios_custom(temp_map):
    """
    Build all 81 (name, temps_sc, S_sc, prob) scenarios using the supplied
    temp_map = {'bad': arr12, 'normal': arr12, 'good': arr12}.
    Survival coupling is unchanged: S_bad_v when stage is 'good' AND
    previous stage was also 'good', otherwise S_normal_v.
    """
    scenarios = []
    for t1 in labels:
        for t2 in labels:
            for t3 in labels:
                for t4 in labels:
                    name = f's1_{t1}__s2_{t2}__s3_{t3}__s4_{t4}'
                    temps_sc = np.zeros(T)
                    S_sc     = np.full(T, S_normal_v)
                    stage_seq = [t1, t2, t3, t4]
                    for i, (sl, months) in enumerate(zip(stage_seq, stage_slices)):
                        prev = stage_seq[i - 1] if i > 0 else None
                        surv = S_bad_v if (sl == 'good' and prev == 'good') else S_normal_v
                        for mm in months:
                            S_sc[mm] = surv
                        temps_sc[months] = _tile(temp_map[sl], len(months))
                    scenarios.append((name, temps_sc, S_sc, 1.0 / 81))
    return scenarios


In [15]:
# WS helper (parameterised by temp_map)

def run_ws_for(units_df_sub, temp_map, mip_gap=0.02):
    scenarios = build_scenarios_custom(temp_map)
    n_feas = 0
    ws_obj = 0.0
    t0 = time.time()

    for sc_name, temps_sc, S_sc, prob in scenarios:
        milp = SalmonFarmingMILP(
            units_df=units_df_sub, temps_t=temps_sc, survival_rates=S_sc,
            horizon_months=T, loc_mab=loc_mab, regional_mab=regional_mab,
            scenario_name=sc_name,
        )
        milp.model.Params.OutputFlag = 0
        milp.model.Params.MIPGap = mip_gap
        milp.model.optimize()
        if milp.model.SolCount > 0:
            n_feas += 1
            ws_obj += prob * milp.model.ObjVal

    return ws_obj, time.time() - t0, n_feas


# EEV helper (parameterised by temp_map)

NA_MONTHS = set(range(0, 45))  # stages 0-2 fixed


def run_eev_for(units_df_sub, temp_map, mip_gap=0.02):
    """
    EEV: EV model uses expected temperatures (average of the three branches —
    equals the normal profile under any symmetric tree). Stage 0-2 decisions
    are fixed and re-evaluated across all 81 custom scenarios.
    """
    # Step 1: EV model — expected temperature = mean(cold, normal, warm)
    temps_exp_12 = (temp_map['bad'] + temp_map['normal'] + temp_map['good']) / 3.0
    temps_exp = np.tile(temps_exp_12, (T // 12) + 1)[:T]
    S_exp = np.full(T, (2.0 * S_normal_v + S_bad_v) / 3.0)

    t0 = time.time()
    ev_m = SalmonFarmingMILP(
        units_df=units_df_sub, temps_t=temps_exp, survival_rates=S_exp,
        horizon_months=T, loc_mab=loc_mab, regional_mab=regional_mab,
        density_limit=25.0, verbose=False,
    )
    ev_m.model.Params.OutputFlag = 0
    ev_m.model.Params.MIPGap = mip_gap
    ev_m.model.optimize()
    if ev_m.model.SolCount == 0:
        print('  EV model infeasible')
        return None, time.time() - t0, 0

    # Step 2: Extract stage 0-2 decisions
    ev_stk = {
        (u, t): round(ev_m.variables['z'][u, t].X)
        for u in ev_m.U for t in ev_m.Tset if t in NA_MONTHS
    }
    ev_harv = {
        (u, s, t): round(ev_m.variables['h'][u, s, t].X)
        for u in ev_m.U for s in ev_m.Tset
        for t in ev_m.H_by_us.get((u, s), []) if t in NA_MONTHS
    }
    ev_hexist = {
        (u, t): round(ev_m.variables['h_exist'][u, t].X)
        for u in ev_m.U_exist for t in ev_m.Tset if t in NA_MONTHS
    }
    ev_q = {
        (u, s): ev_m.variables['q'][u, s].X
        for u in ev_m.U for s in ev_m.Tset
        if s in NA_MONTHS and round(ev_m.variables['z'][u, s].X) == 1
    }

    # Step 3: Fix decisions and evaluate on custom scenarios
    scenarios = build_scenarios_custom(temp_map)
    n_feas = 0
    eev_obj = 0.0
    feas_prob = 0.0

    for sc_name, temps_sc, S_sc, prob in scenarios:
        milp = SalmonFarmingMILP(
            units_df=units_df_sub, temps_t=temps_sc, survival_rates=S_sc,
            horizon_months=T, loc_mab=loc_mab, regional_mab=regional_mab,
            scenario_name=sc_name, disable_economic_presolve=True,
        )
        model = milp.model
        model.update()

        for (u, t), val in ev_stk.items():
            v = model.getVarByName(f'z[{u},{t}]')
            if v is not None:
                v.LB = val; v.UB = val
        for (u, s, t), val in ev_harv.items():
            v = model.getVarByName(f'h[{u},{s},{t}]')
            if v is not None:
                v.LB = val; v.UB = val
        for (u, t), val in ev_hexist.items():
            v = model.getVarByName(f'h_exist[{u},{t}]')
            if v is not None:
                v.LB = val; v.UB = val
        for (u, s), val in ev_q.items():
            v = model.getVarByName(f'q[{u},{s}]')
            if v is not None:
                v.LB = val; v.UB = val

        model.Params.OutputFlag = 0
        model.Params.MIPGap = mip_gap
        model.update()
        model.optimize()

        if model.SolCount > 0:
            n_feas += 1
            feas_prob += prob
            eev_obj += prob * model.ObjVal

    eev_normalized = eev_obj / feas_prob if feas_prob > 0 else 0.0
    return eev_normalized, time.time() - t0, n_feas


In [16]:
# Main experiment: 10 cages, reduced-variance scenario tree, mip_gap=2%

import gurobipy as gb

N_CAGES = 10
MIP_GAP = 0.02

sub_df = full_units_df.head(N_CAGES).reset_index(drop=True)
print(f'Instance: {N_CAGES} cages, reduced-variance temps, mip_gap={MIP_GAP:.0%}')

Instance: 10 cages, reduced-variance temps, mip_gap=2%


In [17]:
# EEV
print('\n--- EEV ---')
eev_obj, eev_time, eev_feas = run_eev_for(sub_df, temp_map_rv, mip_gap=MIP_GAP)
print(f'  EEV = {eev_obj/1e6:.1f} MNOK  |  {eev_time:.1f}s  |  {eev_feas}/81 feasible')


--- EEV ---
  EEV = 692.5 MNOK  |  54.9s  |  81/81 feasible


In [18]:
# SP
print('\n--- SP ---')
ald = BinaryProgressiveHedging(
    units_df=sub_df, loc_mab=loc_mab, regional_mab=regional_mab,
    T=T, mip_gap=MIP_GAP,
    temps_bad=temps_cold_rv, temps_normal=temps_normal_rv, temps_good=temps_warm_rv,
)
ald.build()
ald.solve()
sp_obj  = ald.eval_obj
sp_time = ald.total_time
print(f'  SP  = {sp_obj/1e6:.1f} MNOK  |  {sp_time:.1f}s  |  {ald.n_feasible}/{ald.n_scenarios} feasible')
del ald


--- SP ---
Built 81 scenarios
NA variables: 91060 total (91060 binary)
Variable cache built (tree-aware)


Step 0 (initial solve): 100%|██████████| 81/81 [00:43<00:00,  1.87sc/s]


Step 0 done — E[obj]: 808,309,174
Auto-calibrated rho_bin=8.078e+06, rho_max_bin=8.083e+09

Step 0 | Obj: 808309173.90 | MaxDev: 9.88e-01 | AvgBin: 0.0004 | Fixed: 0
Starting PH iterations (max 1000)
  [Iter 1] progress: 5/81 done, 76 running
  [Iter 1] progress: 10/81 done, 71 running
  [Iter 1] progress: 15/81 done, 66 running
  [Iter 1] progress: 20/81 done, 61 running
  [Iter 1] progress: 25/81 done, 56 running
  [Iter 1] heartbeat: 25/81 done, 56 running, oldest=5.2s | pending: 24,25,26,28,29,30,31,32,33,34,35,36,...
  [Iter 1] progress: 30/81 done, 51 running
  [Iter 1] progress: 35/81 done, 46 running
  [Iter 1] progress: 40/81 done, 41 running
  [Iter 1] progress: 45/81 done, 36 running
  [Iter 1] progress: 50/81 done, 31 running
  [Iter 1] heartbeat: 52/81 done, 29 running, oldest=10.3s | pending: 51,53,54,55,56,57,58,59,60,61,62,63,...
  [Iter 1] progress: 55/81 done, 26 running
  [Iter 1] progress: 60/81 done, 21 running
  [Iter 1] progress: 65/81 done, 16 running
  [Iter 1]

In [19]:
# DE
print('\n--- DE ---')
de_obj = de_time = de_gap = None
try:
    de = DeterministicEquivalent(
        units_df=sub_df, loc_mab=loc_mab, regional_mab=regional_mab,
        T=T, mip_gap=MIP_GAP,
        temps_bad=temps_cold_rv, temps_normal=temps_normal_rv, temps_good=temps_warm_rv,
    )
    de.build()
    de.solve()
    de_obj  = de.obj_val
    de_time = de.solve_time
    de_gap  = de.model.MIPGap if de.model.SolCount > 0 else float('nan')
    print(f'  DE  = {de_obj/1e6:.1f} MNOK  |  {de_time:.1f}s  |  gap {de_gap:.2%}')
except (MemoryError, gb.GurobiError) as exc:
    print(f'  DE  = OOM ({exc})')
finally:
    try:
        de.model.dispose()
    except Exception:
        pass
    del de


--- DE ---
  Prepared 81 scenarios  (3×3×3×3, 4 stages of uncertainty)
  Variables: 0
  Per-scenario constraints: 0
  Total constraints with NAC: 0
Set parameter MIPGap to value 0.02

DE model built in 94.1s  | 0 vars  | 0 constraints

--- Solving DE ---
Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (mac64[arm] - Darwin 25.3.0 25D2128)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 3380715 rows, 2194371 columns and 22202838 nonzeros
Model fingerprint: 0xe30a4ca1
Variable types: 1567431 continuous, 578340 integer (578340 binary)
Semi-Variable types: 48600 continuous, 0 integer
Coefficient statistics:
  Matrix range     [1e-01, 2e+07]
  Objective range  [6e-04, 2e+05]
  Bounds range     [1e+00, 3e+06]
  RHS range        [1e+00, 4e+07]
Presolve removed 1321051 rows and 1583382 columns (presolve time = 5s) ...
Presolve removed 2812481 rows and 1927369 columns
Presolve time: 7.32s
Presolved: 593020 rows, 284255 col

In [20]:
# WS
print('\n--- WS ---')
ws_obj, ws_time, ws_feas = run_ws_for(sub_df, temp_map_rv, mip_gap=MIP_GAP)
print(f'  WS  = {ws_obj/1e6:.1f} MNOK  |  {ws_time:.1f}s  |  {ws_feas}/81 feasible')


--- WS ---
  WS  = 808.4 MNOK  |  213.6s  |  81/81 feasible


In [21]:
# Derived metrics
vss_sp  = (sp_obj - eev_obj) / 1e6
vss_de  = (de_obj - eev_obj) / 1e6 if de_obj is not None else None
evpi_sp = (ws_obj - sp_obj)  / 1e6
evpi_de = (ws_obj - de_obj)  / 1e6 if de_obj is not None else None
print(f'  VSS_SP={vss_sp:.1f}  VSS_DE={vss_de}  EVPI_SP={evpi_sp:.1f}  EVPI_DE={evpi_de}')

  VSS_SP=43.1  VSS_DE=59.7948223037312  EVPI_SP=72.8  EVPI_DE=56.072298498764276


In [22]:
# Results table

def _fmt(val, suffix=''):
    return f'{val:.1f}{suffix}' if val is not None else 'N/A'

rows = [
    {'Metric': 'Scenarios',              'EEV': f'{eev_feas}/81*', 'SP': '81/81', 'DE': '81', 'WS': '81'},
    {'Metric': 'E[obj] (MNOK)',           'EEV': _fmt(eev_obj/1e6), 'SP': _fmt(sp_obj/1e6), 'DE': _fmt(de_obj/1e6) if de_obj else 'N/A', 'WS': _fmt(ws_obj/1e6)},
    {'Metric': 'MIP gap tolerance (%)',   'EEV': '2', 'SP': '2', 'DE': '2', 'WS': '2'},
    {'Metric': 'Wall clock time (s)',     'EEV': _fmt(eev_time), 'SP': _fmt(sp_time), 'DE': _fmt(de_time) if de_time else 'N/A', 'WS': _fmt(ws_time)},
    {'Metric': 'VSS_SP (MNOK)',           'EEV': '', 'SP': '', 'DE': '', 'WS': _fmt(vss_sp)},
    {'Metric': 'VSS_DE (MNOK)',           'EEV': '', 'SP': '', 'DE': '', 'WS': _fmt(vss_de) if vss_de is not None else 'N/A'},
    {'Metric': 'EVPI_SP (MNOK)',          'EEV': '', 'SP': '', 'DE': '', 'WS': _fmt(evpi_sp)},
    {'Metric': 'EVPI_DE (MNOK)',          'EEV': '', 'SP': '', 'DE': '', 'WS': _fmt(evpi_de) if evpi_de is not None else 'N/A'},
]
df_results = pd.DataFrame(rows).set_index('Metric')
display(df_results)
print(f'\n* EEV expectation over {eev_feas} feasible scenarios (prob mass {eev_feas/81:.3f})')


,EEV,SP,DE,WS
Metric,,,,
Scenarios,81/81*,81/81,81,81
E[obj] (MNOK),692.5,735.6,752.3,808.4
MIP gap tolerance (%),2,2,2,2
Wall clock time (s),54.9,197.0,3329.9,213.6
VSS_SP (MNOK),,,,43.1
VSS_DE (MNOK),,,,59.8
EVPI_SP (MNOK),,,,72.8
EVPI_DE (MNOK),,,,56.1



* EEV expectation over 81 feasible scenarios (prob mass 1.000)
